# POC 2: The Escalation Calibrator

**Pain point:** A client escalates. Before you respond, you need to know: was this actually a breach? What did the contract say? What was delivered vs promised? Without grounding, you either over-apologise (owning something you didn't cause) or dismiss something you should own — both damage trust.

**What this notebook shows:** Grounding with contract terms and delivery records lets the model produce a factual promise-vs-reality diff. The ungrounded model gives generic de-escalation advice that could make things worse.

**You need:** A free Groq API key from https://console.groq.com

**Connecting Dots**, is where I write about the patterns I notice while building, checkout my blogs for more

👉 https://sriharshacr.github.io/blogs/

In [ ]:
!pip install groq -q

In [2]:
import os
from groq import Groq

In [3]:
MODEL = 'qwen/qwen3.8-27b'

In [4]:
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY') or input('Enter Groq API key: ')

client = Groq(api_key=GROQ_API_KEY)

In [ ]:
def call_llm(system_prompt, user_message, temperature=0.2, max_tokens=900):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

print(f'Model ready: {MODEL}')

In [ ]:
# --- Synthetic grounding material ---

CONTRACT_SLA = """
CONTRACT EXCERPT — Acme Corp / TechPartner Ltd
Effective: 1 Jan 2025

Section 4.1 — Reporting Deliverables:
  Quarterly performance reports to be delivered within 5 business days of quarter close.
  Monthly summary updates are provided on a best-efforts basis and are not contractually guaranteed.

Section 4.2 — Bug Resolution SLAs:
  Critical severity (P1): resolution or workaround within 48 hours of confirmed report.
  High severity (P2): resolution within 5 business days.
  Medium/Low (P3/P4): next scheduled release.

Section 4.3 — Scope:
  Support covers the production environment only.
  Pre-production environment support is outside scope of this agreement.
"""

DELIVERY_RECORDS = """
DELIVERY LOG — Q2 & Q3 2025

Q2 Quarterly Report:
  Due: 7 Jul 2025 (5 business days after 30 Jun)
  Delivered: 4 Jul 2025 — 3 days early
  Client acknowledgement: received, no issues raised

Q3 Quarterly Report:
  Due: 7 Oct 2025 (5 business days after 30 Sep)
  Delivered: 10 Oct 2025 — 3 days late
  Client notified in advance: yes, email sent 2 Oct citing data pipeline issue
  Client response to advance notice: 'Understood, please send ASAP'

Bug #247 — P1 Critical:
  Reported by client: 5 Sep 2025, 09:00
  Resolved: 8 Sep 2025, 11:00 — 74 hours after report (SLA: 48h) — BREACH: 26h over
  Root cause: on-call engineer unavailable; backup cover gap
  Client notified of delay: no

Monthly updates (Jul, Aug, Sep 2025):
  Not sent — monthly updates are best-efforts per Section 4.1, not contractually required
"""

ESCALATION_EMAIL = """
FROM: James Whitfield, CTO, Acme Corp
TO: Account Manager, TechPartner Ltd
DATE: 15 Oct 2025
SUBJECT: Formal Escalation — Service Level Failures

We are formally escalating our concerns regarding service delivery over the past quarter.

Specifically:
1. The Q3 quarterly report was delivered late — this is unacceptable given the contractual commitment.
2. A critical bug (Bug #247) took three days to resolve. We were promised 48-hour resolution.
3. We have not received a single monthly update since July despite our repeated requests.

We are requesting a formal response within 48 hours and a remediation plan.
If these issues are not addressed, we will escalate to the contract review process under Section 8.
"""

FULL_CONTEXT = f"""
=== CONTRACT / SLA TERMS ===
{CONTRACT_SLA}

=== DELIVERY RECORDS ===
{DELIVERY_RECORDS}

=== CLIENT ESCALATION EMAIL ===
{ESCALATION_EMAIL}
"""

print('Grounding data loaded.')

In [ ]:
# --- UNGROUNDED call ---

ungrounded_system = "You are a helpful customer success assistant."

ungrounded_query = """
A client has sent a formal escalation saying we delivered a quarterly report late,
a critical bug took 3 days instead of 48 hours, and we haven't sent monthly updates.
How should I respond?
"""

print('=== UNGROUNDED OUTPUT ===')
print(call_llm(ungrounded_system, ungrounded_query))

In [ ]:
# --- GROUNDED call ---

grounded_system = """
You are a precise contract and delivery analyst. Your job is to produce a factual
'promise vs. reality' diff before any response is drafted.

For each item in the escalation:
  - State exactly what was contractually committed
  - State exactly what was delivered (with dates)
  - Verdict: BREACH, PARTIAL BREACH, NOT A BREACH, or NOT IN CONTRACT
  - One-line suggested position for the response (own it / clarify it / correct the record)

Be precise. Cite section numbers and dates. Do not speculate.
Do not draft the full response — only the factual diff.
"""

grounded_query = f"""
We have received a formal escalation. Analyse each claim against our contract and delivery records.

{FULL_CONTEXT}
"""

print('=== GROUNDED OUTPUT: Promise vs Reality Diff ===')
print(call_llm(grounded_system, grounded_query))

In [ ]:
# --- BONUS: Draft the calibrated response using the diff ---
# Run this after reviewing the diff above.

diff_result = call_llm(
    grounded_system,
    f"We have received a formal escalation. Analyse each claim.\n{FULL_CONTEXT}"
)

response_system = """
You are a senior account manager drafting a formal escalation response.
Use the factual diff provided. Own genuine breaches clearly and briefly.
Correct the record on items that are not breaches — factually, not defensively.
Propose one concrete remediation action for each genuine breach.
Tone: direct, professional, no over-apologising, no dismissiveness.
Length: under 200 words.
"""

response_query = f"""
Based on this factual analysis, draft the formal response to the client:

{diff_result}

Original escalation for reference:
{ESCALATION_EMAIL}
"""

print('=== CALIBRATED RESPONSE DRAFT ===')
print(call_llm(response_system, response_query, max_tokens=400))

## What just happened

The **ungrounded** model gives generic de-escalation advice — apologise, acknowledge, propose a call. It has no idea that one of the three claims isn't actually in the contract (monthly updates are best-efforts), or that the Q3 report delay was notified in advance.

The **grounded** model produces a factual diff:
- **Q3 report:** partial breach (late, but client was notified in advance — mitigating factor)
- **Bug #247:** genuine breach — 74h vs 48h SLA, client was not notified
- **Monthly updates:** not in contract — client expectation, not a contractual commitment

That diff is what the account manager needed before picking up the phone.

**Note on grounding temperature:** This notebook uses `temperature=0.2` — lower than other POCs. For factual extraction tasks where hallucination is costly, lower temperature is worth the slight reduction in fluency.